In [50]:
#julia version:1.11.6
include("Stability_Cavity.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using DelimitedFiles
using PyCall
using BSplineKit

In [51]:
function eigsol(F,G,H,T,R,omega,be,N_cheb,D,D2,c,num)
    sigma = 0.72
    cof = CRC_STA.Spatial_mode_BEK(F,-G,H,T,sigma,N_cheb,D,D2,R)
    A0_raw,A1_raw,A2_raw = CRC_STA.assemble_mat(cof :: CRC_STA.COF,D,D2,be,omega,R)
    A0,A1,A2 = CRC_STA.boudary_condition(A0_raw,A1_raw,A2_raw,N_cheb)
    nep = PEP([A0,A1,A2]); 
    eigval,eigvec = iar(nep,σ = c , neigs = num ,maxit = 500,tol=1e-10)
    # eigval = conj(eigval)
    return eigval
end

eigsol (generic function with 1 method)

In [52]:
function Cheb(N)
    θ = range(0,length=N+1,stop=pi)
    x = reshape(-cos.(θ), N+1, 1)
    c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
    X = repeat(x, 1, N+1);
    dX = X - X';
    D = (c * (1 ./ c)') ./ (dX .+ I(N+1));
    D = D - diagm(vec(sum(D, dims=2))); 
    a = 2
    b = 0.6
    c = 0.5
    for i=1:N+1
        D[i,:]=D[i,:].* (1-b*x[i]-(1-b)*(x[i]^3+c*(1-x[i]^2)))^2/(2a*(b .+ 3 * (1-b)*x[i]^2 - 2 * c * (1-b) * x[i]))
    end
    for i=1:N+1
        x[i] = a * (1+b*x[i]+(1-b)*(x[i]^3+c*(1-x[i]^2)))/(1-b*x[i]-(1-b)*(x[i]^3+c*(1-x[i]^2)))
        if x[i] > 20
            x[i] = 20
        end
    end
    D2 = D^2;
    return D,D2,x
 end
 function interp(u0,v0,w0,T0,x,N)
    F = Base.zeros(N+1,1)
    G = Base.zeros(N+1,1)
    H = Base.zeros(N+1,1)
    T = Base.zeros(N+1,1)
    z = range(0,20,2000)
    itu = BSplineKit.interpolate(z, u0 , BSplineOrder(4))
    itv = BSplineKit.interpolate(z, v0 , BSplineOrder(4))
    itw = BSplineKit.interpolate(z, w0 , BSplineOrder(4))
    itt = BSplineKit.interpolate(z, T0 , BSplineOrder(4))
    for i = 1 : N + 1
        F[i,1] = itu(x[i])
        G[i,1] = itv(x[i])
        H[i,1] = itw(x[i])
        T[i,1] = itt(x[i])
    end
    return F,G,H,T
end

interp (generic function with 1 method)

In [53]:
pushfirst!(PyVector(pyimport("sys")."path"), @__DIR__)
bone = pyimport("Bone")
z, H, F, G, T, dF, dG, dT, info = bone.get_baseflow(0.8)
N_cheb = 199
D,D2,x = Cheb(N_cheb)
F,G,H,T = interp(F,G,H,T,x,N_cheb)

([0.0; 0.00017351417635306447; … ; -2.2971629560035864e-26; -2.2971629560035864e-26;;], [0.0; 0.00019036979887964133; … ; 1.0; 1.0;;], [1.5437221420852696e-23; -4.75707694399895e-8; … ; -1.2138285203861623; -1.2138285203861623;;], [0.7999999999999998; 0.8000210484351035; … ; 0.9999999999999999; 0.9999999999999999;;])

In [35]:
# 重新加载 Stability_Cavity.jl（修改后必须执行）
include("Stability_Cavity.jl")
println("✅ CRC_STA 模块已重载")

✅ CRC_STA 模块已重载


In [ ]:
# 强制重载 Bone.py 模块（修改 .py 文件后必须执行）
pyimport("importlib").reload(bone)

In [ ]:
plot(x,F)
plot!(x,-G)
plot!(x,H)
# plot!(x,T)

In [64]:
R_ini = 285.36
omega = 0.0
be_ini = 0.07759
c_ini = 0.4
Tw = 0.98
num = 1
pushfirst!(PyVector(pyimport("sys")."path"), @__DIR__)
bone = pyimport("Bone")
z, H, F, G, T, dF, dG, dT, info = bone.get_baseflow(Tw)
N_cheb = 99
D,D2,x = Cheb(N_cheb)
F,G,H,T = interp(F,G,H,T,x,N_cheb)
eigval = eigsol(F,G,H,T,R_ini,omega,be_ini,N_cheb,D,D2,c_ini,1)

1-element Vector{ComplexF64}:
 0.2599182123792072 + 0.09563162625149774im

In [ ]:
R_ini = 500
omega = 0.0
be_ini = 0.03
c_ini = 0.05
Tw = 1.04
num = 2
pushfirst!(PyVector(pyimport("sys")."path"), @__DIR__)
bone = pyimport("Bone")
z, H, F, G, T, dF, dG, dT, info = bone.get_baseflow(Tw)
N_cheb = 99
D,D2,x = Cheb(N_cheb)
F,G,H,T = interp(F,G,H,T,x,N_cheb)
cur(Tw,omega,R_ini,c_ini,be_ini,num)

InterruptException: InterruptException:

In [46]:
function cur(Tw,omega,R_ini,c_ini,be_ini,num)
    be_step = -0.0005
    pushfirst!(PyVector(pyimport("sys")."path"), @__DIR__)
    bone = pyimport("Bone")
    z, H, F, G, T, dF, dG, dT, info = bone.get_baseflow(Tw)
    N_cheb = 99
    D,D2,x = Cheb(N_cheb)
    F,G,H,T = interp(F,G,H,T,x,N_cheb)
    initial = []
    tempvec_1 = [0 0 0 0 0 0 0]
    eigval = 0
    writedlm("output.dat",initial)
    writedlm("output_eig.dat",initial)
    eigval_ori = eigsol(F,G,H,T,R_ini,omega,be_ini,N_cheb,D,D2,c_ini,num)
    open("output_eig.dat", "a") do io
        write(io,"be=$be_ini,eig=$eigval\n")
    end
    eigval = sort(eigval_ori, by=real)
    if imag(eigval_ori[1]) < 0
        for be = be_ini :  be_step : -0.5
            sig_last1 = sign(imag(eigval[1]))
            sig_last2 = sign(imag(eigval[end]))
            eigval = eigsol(F,G,H,T,R_ini,omega,be,N_cheb,D,D2,c_ini,num)
            sig_now1 = sign(imag(eigval[1]))
            sig_now2 = sign(imag(eigval[end]))
            # point = filter(x -> abs(imag(x)) < 0.0004 , eigval)
            open("output_eig.dat", "a") do io
                write(io,"be=$be,eig=$eigval\n")
            end
            if sig_last1 * sig_now1 < 0 || sig_last2 * sig_now2 < 0
                initial = [omega R_ini be real(eigval[1]) imag(eigval[1]) real(eigval[end]) imag(eigval[end])]
                break
            end
        end
    elseif imag(eigval_ori[1]) > 0
        for be = be_ini : - be_step : 0.5
           sig_last1 = sign(imag(eigval[1]))
            sig_last2 = sign(imag(eigval[end]))
            eigval = eigsol(F,G,H,T,R_ini,omega,be,N_cheb,D,D2,c_ini,num)
            sig_now1 = sign(imag(eigval[1]))
            sig_now2 = sign(imag(eigval[end]))
            # point = filter(x -> abs(imag(x)) < 0.0004 , eigval)
            open("output_eig.dat", "a") do io
                write(io,"be=$be,eig=$eigval\n")
            end
            if sig_last1 * sig_now1 < 0 || sig_last2 * sig_now2 < 0
                initial = [omega R_ini be real(eigval[1]) imag(eigval[1]) real(eigval[end]) imag(eigval[end])]
                break
            end
        end
    end
    total = initial
    be = initial[3] - be_step
    dir = 0
    boundlen = 3
 # CACULATE

    while true
        index = findall(x->x==findmin([total[end,5],total[end,7]])[1],total[end,:])
        c = total[end,index[1] - 1]
        eigval = eigsol(F,G,H,T,total[end,2],omega,be,N_cheb,D,D2,c,num)
        eigval_1 = eigsol(F,G,H,T,total[end,2],omega,be-0.001,N_cheb,D,D2,c,num)
        eigval_2 =  eigsol(F,G,H,T,total[end,2],omega,be+0.001,N_cheb,D,D2,c,num)
        index1 = findmin(x-> (imag(x)) , eigval_1)[2]
        index2 = findmin(x-> (imag(x)) , eigval_2)[2]
        num = 1
        if size(total,1) > 3 && abs(total[end,2] - total[end-1,2]) <=2
            R_step = 0.25
        else
            R_step = 1
        end
        if (imag(eigval_1[index1]) < 0 && imag(eigval_2[index2]) > 0) || (imag(eigval_1[index1]) > 0 && imag(eigval_2[index2]) > 0) || dir == -1
            mode = 1
        elseif (imag(eigval_1[index1]) > 0 && imag(eigval_2[index2]) < 0) || (imag(eigval_1[index1]) < 0 && imag(eigval_2[index2]) < 0)
            mode = 2
        end
        
        if mode == 1 

            for R = total[end,2] : R_step : 700

                eigval = eigsol(F,G,H,T,R,omega,be,N_cheb,D,D2,c,num)
                index = findmin((imag),eigval)[2]

                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1]) real(eigval[end] ) imag(eigval[end])]]     
                len = size(tempvec_1,1)
                open("output.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if ((tempvec_1[end-1, 5] * tempvec_1[end,5]) < 0 && abs(tempvec_1[end,5] < 0.001)) || (abs(tempvec_1[end,5])<3e-5) || ((tempvec_1[end-1, 7] * tempvec_1[end,7]) < 0) || (abs(tempvec_1[end,7])<3e-5) 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end

                if (len > boundlen && abs(tempvec_1[end,5]) > abs(tempvec_1[end-1,5])) || len > 100

                    mode = 2
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
            end        
        end


        if mode == 2

            for R = total[end,2]: -R_step : 0
                
                eigval = eigsol(F,G,H,T,R,omega,be,N_cheb,D,D2,c,num)
                index = findmin((imag),eigval)[2]
                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1]) real(eigval[end] ) imag(eigval[end])]]     
                len = size(tempvec_1,1)
                open("output.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end
                if ((tempvec_1[end-1, 5] * tempvec_1[end,5]) < 0 && abs(tempvec_1[end,5] < 0.001)) || (abs(tempvec_1[end,5])<3e-5) || ((tempvec_1[end-1, 7] * tempvec_1[end,7]) < 0) || (abs(tempvec_1[end,7])<3e-5) 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
                
                if (len > boundlen && abs(tempvec_1[end,5]) > abs(tempvec_1[end-1,5])) || len > 100

                    mode = 1
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
            end        
        end

        if mode == 1

            for R = total[end,2]: R_step : 700

                if total[end,3] == be

                    break

                end 
                
                eigval = eigsol(F,G,H,T,R,omega,be,N_cheb,D,D2,c,num)
                index = findmin((imag),eigval)[2]
                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1]) real(eigval[end] ) imag(eigval[end])]]     

                len = size(tempvec_1,1)
                open("output.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if ((tempvec_1[end-1, 5] * tempvec_1[end,5]) < 0 && abs(tempvec_1[end,5] < 0.001)) || (abs(tempvec_1[end,5])<3e-5) || ((tempvec_1[end-1, 7] * tempvec_1[end,7]) < 0) || (abs(tempvec_1[end,7])<3e-5) 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
                
                if (len > boundlen && abs(tempvec_1[end,5]) > abs(tempvec_1[end-1,5])) || len > 100

                    mode = 2
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
            end        
        end
        c = total[end,4]
        eigval = eigsol(F,G,H,T,total[end,2] + 2,omega,be,N_cheb,D,D2,c,num)
        sig = findmin(imag,eigval)[1]
        if size(total,1) > 10 &&total[end,3] != be 
            dir = -1 
            be_step = 0.0009
        else
            if sig > 0
                dir = -1
                be_step = 0.00075
            else
                dir = 1
                be_step = 0.00075
            end
        end
            be += 0.0008
        filename = "ome=$(omega)_Tw=$(Tw).dat"
        str1 = "Variables=\"omega\" \"R\" \"beta\" \"alpha_r_1\" \"alpha_i_1\" \"alpha_r_2\" \"alpha_i_2\""
        str2 = "Zone T=\"omega=$(omega),Tw=$(Tw)\""
        open(filename,"w") do io
            println(io,str1)
            println(io,str2)
            writedlm(io,total[2:end,:])
        end
        if total[end,2] > 500 && size(total,1) > 30 
            break
        end
        # app = readdlm("ome=$(omega)_Tw=$(Tw)_Mr=$(Mr)_temp.dat")
        # ori = readdlm("ome=$(omega)_Tw=$(Tw)_Mr=$(Mr).dat")
        # if app[end,2] < ori[3,2]
        #     break
        # end
    end
end

cur (generic function with 1 method)

In [ ]:
# 重新加载 Stability_Cavity.jl（修改后必须执行）
include("Stability_Cavity.jl")
println("✅ CRC_STA 模块已重载")

In [54]:
# ═══ 验证等温中性点 ═══
# 使用 N_cheb=199, BSplineKit 插值
z_iso, H_iso, F_iso, G_iso, T_iso, dF_iso, dG_iso, dT_iso, info_iso = bone.get_baseflow(1.0)
F_iso, G_iso, H_iso, T_iso = interp(F_iso, G_iso, H_iso, T_iso, x, N_cheb)
println("Tw=1.0 基本流就绪, N_cheb=$N_cheb")
println("F'(0)≈$( (F_iso[2]-F_iso[1])/(x[2]-x[1]) ), G'(0)≈$( (G_iso[2]-G_iso[1])/(x[2]-x[1]) ), H(∞)=$(H_iso[end])")

# 文献中的时间模式中性点: R=285.36, β=0.07759
R_test = 285.36
beta_test = 0.07759
e = eigsol(F_iso, G_iso, H_iso, T_iso, R_test, 0.0, beta_test, N_cheb, D, D2, 0.4, 1)
println("\nR=$R_test, β=$beta_test:")
println("α = $(e[1])")
println("-Im(α) = $(-imag(e[1]))")

# 也测试稍微不同的 β
for be in [0.077, 0.07759, 0.078, 0.079]
    e2 = eigsol(F_iso, G_iso, H_iso, T_iso, R_test, 0.0, be, N_cheb, D, D2, 0.4, 1)
    println("β=$be: Im(α)=$(imag(e2[1]))")
end

Tw=1.0 基本流就绪, N_cheb=199
F'(0)≈0.5100954229560987, G'(0)≈0.615922433858873, H(∞)=-0.884473414805651

R=285.36, β=0.07759:
α = 0.3850525227249444 + 0.000348653593682377im
-Im(α) = -0.000348653593682377
β=0.077: Im(α)=0.0003616323083264889
β=0.07759: Im(α)=0.0003486576036238065
β=0.078: Im(α)=0.00035976101548354164
β=0.079: Im(α)=0.0004519181368758674


In [55]:
# ═══════════════════════════════════════════════════════════════
#  壁温对稳定性的影响分析 (BSplineKit 插值, N_cheb=199)
#  ═══════════════════════════════════════════════════════════════

Tws = [0.8, 0.9, 0.95, 0.98, 1.0, 1.02, 1.04]
Tws_dict = Dict{Float64, Tuple{Matrix{Float64},Matrix{Float64},Matrix{Float64},Matrix{Float64}}}()
for Tw in Tws
    zz, HH, FF, GG, TT, dFF, dGG, dTT, info = bone.get_baseflow(Tw)
    if info["success"]
        Fb, Gb, Hb, Tb = interp(FF, GG, HH, TT, x, N_cheb)
        Tws_dict[Tw] = (Fb, Gb, Hb, Tb)
        println("Tw=$Tw: H∞=$(Hb[end])")
    end
end
println("✅ 基本流就绪")

R_test = 285.36
betas = [0.06, 0.07, 0.07759, 0.09]
println("\n=== 等温中性点附近 β 扫描 (R=285.36) ===")
println("Tw\t\tβ=0.06\t\tβ=0.07\t\tβ=0.07759\tβ=0.09")
for Tw in Tws
    if !haskey(Tws_dict, Tw); continue; end
    Fb, Gb, Hb, Tb = Tws_dict[Tw]
    print("$(Tw)")
    for be in betas
        e = eigsol(Fb, Gb, Hb, Tb, R_test, 0.0, be, N_cheb, D, D2, 0.4, 1)
        print("\t$(round(imag(e[1]), digits=5))")
    end
    println()
end

Tw=0.8: H∞=-1.2138285203861623
Tw=0.9: H∞=-1.0898391861963737
Tw=0.95: H∞=-1.0050898445182737
Tw=0.98: H∞=-0.9394702201483134
Tw=1.0: H∞=-0.884473414805651
Tw=1.02: H∞=-0.8123406189807595
Tw=1.04: H∞=-0.6948568193458373
✅ 基本流就绪

=== β 扫描 (R=285.36) ===
Tw\β	β=0.06	β=0.065	β=0.07	β=0.075	β=0.08	β=0.085	β=0.09	β=0.095	β=0.1
0.8	-0.18865	-0.19541	-0.1967	-0.19158	-0.18111	-0.16776	0.18098	0.18576	0.08989
0.9	-0.20617	-0.18467	-0.16912	-0.15706	-0.14718	-0.13877	-0.13139	-0.12475	-0.11868
0.95	0.01961	0.02966	0.03883	0.04733	0.05531	0.06286	0.07006	0.10702	0.11064
0.98	0.18305	0.19149	0.19959	0.09392	0.09721	0.10044	0.10362	0.10676	0.10987
1.0	0.02405	0.01154	0.00385	0.00068	0.00063	0.00256	0.00574	0.00971	0.01417
1.02	0.03766	0.04276	0.04776	0.05272	0.05769	0.0627	0.06779	0.07296	0.07824
1.04	-0.00991	-0.00319	0.00333	0.00969	0.01591	0.022	0.02799	0.0339	0.03972


In [ ]:
plot(abs.(eigvec[:,1]))